<a href="https://colab.research.google.com/github/vitor-hk/Processamento-de-Linguagem-Natural/blob/main/extracao_licitacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 73.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import requests
from bs4 import BeautifulSoup
import re
import spacy
import xml.etree.ElementTree as ET
from xml.dom import minidom
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
nlp = spacy.load("pt_core_news_sm")
stop_words_pt = set(stopwords.words('portuguese'))

class PipelineDOE:
    def __init__(self, url_base):
        self.url_base = url_base



    def coletar_textos_brutos(self):
        print("[1] Acessando o Diário Municipal e varrendo páginas...")
        headers = {'User-Agent': 'Mozilla/5.0'}
        textos_extraidos = []
        pagina = 1

        while True:
            if pagina == 1:
                url = self.url_base
            else:
                url = f"{self.url_base}&AtoASolrDocument_page={pagina}"

            print(f"  -> Coletando página {pagina}...")

            try:
                resposta = requests.get(url, headers=headers)
                resposta.raise_for_status()
                soup = BeautifulSoup(resposta.text, 'html.parser')

                titulos_h4 = soup.find_all('h4')

                if not titulos_h4:
                    print("  -> Fim das páginas alcançado.")
                    break

                for h4 in titulos_h4:
                    container = h4.parent

                    if len(container.find_all('h4')) > 1:
                        linhas_texto = [h4.get_text(strip=True)]
                        for irmao in h4.find_next_siblings():
                            if irmao.name == 'h4':
                                break
                            linhas_texto.append(irmao.get_text(separator=' ', strip=True))
                        texto = "\n".join(linhas_texto).strip()
                    else:
                        texto = container.get_text(separator='\n').strip()

                    if texto and texto not in textos_extraidos:
                        textos_extraidos.append(texto)

                pagina += 1

            except Exception as e:
                print(f"Erro na coleta da página {pagina}: {e}")
                break

        print(f"✔️ {len(textos_extraidos)} publicações totais coletadas em todas as páginas!")
        return textos_extraidos

    def pre_processar_texto(self, texto):
        texto_limpo = re.sub(r'[^\w\s]', '', texto.lower())
        tokens = word_tokenize(texto_limpo, language='portuguese')
        tokens_sem_stop = [w for w in tokens if w not in stop_words_pt]
        return " ".join([t.lemma_ for t in nlp(" ".join(tokens_sem_stop))])

    def eh_edital_licitacao(self, texto):
        chaves = ["edital", "pregão", "tomada", "concorrência", "termo", "dispensa", "homologação"]
        return any(chave in texto.lower() for chave in chaves)


    def extrair_dados(self, texto):
        dados = {"titulo": "", "informacao": "", "data": "", "assinado_por": ""}
        linhas = [l.strip() for l in texto.split('\n') if l.strip()]

        if not linhas: return dados
        dados["titulo"] = linhas[0]

        match_data = re.search(r"(\d{2}/\d{2}/\d{4})", texto)
        if match_data:
            data_bruta = match_data.group(1)
            meses = {"01":"janeiro", "02":"fevereiro", "03":"março", "04":"abril", "05":"maio", "06":"junho", "07":"julho", "08":"agosto", "09":"setembro", "10":"outubro", "11":"novembro", "12":"dezembro"}
            try:
                dia, mes, ano = data_bruta.split('/')
                dados["data"] = f"{dia} de {meses[mes]} de {ano}"
            except:
                dados["data"] = data_bruta

        linhas_limpas = []
        for linha in linhas[1:]:
            if "[Imprimir Extrato]" in linha or "[Abrir/Salvar Original]" in linha or re.match(r"^\d{2}/\d{2}/\d{4}", linha):
                continue
            linhas_limpas.append(linha)

        info_texto = " ".join(linhas_limpas)
        info_texto = re.sub(r'\s+', ' ', info_texto).strip()
        dados["informacao"] = info_texto

        doc = nlp(info_texto)
        assinantes = [ent.text for ent in doc.ents if ent.label_ == "PER"]
        dados["assinado_por"] = assinantes[-1] if assinantes else "Não identificado (Requer documento original)"

        return dados

    def gerar_xml_lote(self, lista_dados):
        print("[5] Gerando arquivo XML final...")
        root = ET.Element("licitacoes")

        for d in lista_dados:
            lic = ET.SubElement(root, "licitacao")
            ET.SubElement(lic, "titulo").text = d["titulo"]
            ET.SubElement(lic, "informacao").text = d["informacao"]
            ET.SubElement(lic, "data").text = d["data"]
            ET.SubElement(lic, "assinado_por").text = f"{d['assinado_por']}\n  "

        xml_str = minidom.parseString(ET.tostring(root, encoding='utf-8')).toprettyxml(indent="  ")
        xml_str = '\n'.join([linha for linha in xml_str.split('\n') if linha.strip()])

        xml_str = xml_str.replace("\n  ", "\n\n  ")

        with open("licitacoes_separadas.xml", "w", encoding="utf-8") as f:
            f.write(xml_str)
        return xml_str


if __name__ == "__main__":
    url_base_blumenau = "https://diariomunicipal.sc.gov.br/?r=site/portal&codigoEntidade=41&categoria=Licita%C3%A7%C3%B5es&dataInicial=25%2F08%2F2026&dataFinal=14%2F09%2F2026"

    pipeline = PipelineDOE(url_base_blumenau)
    brutos = pipeline.coletar_textos_brutos()
    dados_finais = []

    if brutos:
        print("[2, 3 e 4] Extraindo dados de todas as páginas...")
        for texto in brutos:
            if pipeline.eh_edital_licitacao(texto):
                dados_finais.append(pipeline.extrair_dados(texto))

        if dados_finais:
            pipeline.gerar_xml_lote(dados_finais)
            print(f"\n✔️ SUCESSO: Arquivo salvo com {len(dados_finais)} itens isolados, de todas as páginas!")
        else:
            print(" Nenhuma publicação passou no filtro.")

[1] Acessando o Diário Municipal e varrendo páginas...
  -> Coletando página 1...
  -> Coletando página 2...
  -> Coletando página 3...
  -> Coletando página 4...
  -> Coletando página 5...
  -> Coletando página 6...
  -> Coletando página 7...
  -> Coletando página 8...
  -> Coletando página 9...
  -> Coletando página 10...
  -> Coletando página 11...
  -> Coletando página 12...
  -> Coletando página 13...
  -> Coletando página 14...
  -> Coletando página 15...
  -> Coletando página 16...
  -> Coletando página 17...
  -> Coletando página 18...
  -> Coletando página 19...
  -> Coletando página 20...
  -> Coletando página 21...
Erro na coleta da página 21: 404 Client Error: Not Found for url: https://diariomunicipal.sc.gov.br/?r=site/portal&codigoEntidade=41&categoria=Licita%C3%A7%C3%B5es&dataInicial=25%2F08%2F2026&dataFinal=14%2F09%2F2026&AtoASolrDocument_page=21
✔️ 67 publicações totais coletadas em todas as páginas!
[2, 3 e 4] Extraindo dados de todas as páginas...
[5] Gerando arquivo